In [15]:

# Uppdaterar ändringar i src utan att behöva starta om kernel:

%load_ext autoreload 
%autoreload 2
%matplotlib inline


import sys
import os
import time

# Check kernel
print(f'path: {sys.executable}')

from typing import Tuple
from pathlib import Path
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split

from heston_project.models.DDN import *
from heston_project.utils import *


# Device Configuration
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# Reproducibility 
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)
if device.type == 'cuda':
    torch.cuda.manual_seed(SEED)

print("Setup Complete.")

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
path: /Users/manswestman/Kandidatarbete/venv/bin/python
Using device: cpu
Setup Complete.


In [16]:
df = pd.read_csv('../../data/full_dataset_training_200000(2).csv')
# df = pd.read_csv(project_root/'data'/'dataset_100K_nofeller.csv')

# Shuffle with a fixed random state for reproducibility
df = df.sample(frac=1, random_state=42).reset_index(drop=True)

df_scaler = df.copy()

#Scale theta & v0 (Ska dessa behållas i vanlig form också för att inte scalea om i output)
df_scaler[['theta', 'v0']] = np.sqrt(df_scaler[['theta', 'v0']])
sqrt_copy = df_scaler[['theta', 'v0']].copy()
df_scaler['pdivk'] = df_scaler['price'] / df_scaler['K']

inputs = df_scaler[['kappa', 'theta', 'v0', 'rho', 'sigma', 'r', 'lm', 'T']]
labels = df_scaler[['pdivk', 'dtheta', 'dsigma', 'drho', 'dkappa', 'dv0']]

# See utils for param bounds

# param_bounds = {
#     "lm":    [-1.5, 1.5],
#     "r":     [-0.01, 0.10],
#     "T":     [1/52, 3.0],
#     "theta": [0.0, 1.0],
#     "sigma": [0.1, 1.5],
#     "rho":   [-0.95, 0.0],
#     "kappa": [0.05, 5.0],
#     "v0":    [0.0, 1.0],
# }

# Skalar inputs till [-1,1]
inputs = scale_inputs(inputs, zero_centered=True)


# Skalar greek targets till dP_tilde/dx (där x=inputs skalat till [-1,1]) genom kedjeregeln
def scale_greek_target(diff, xmin, xmax, K):
    return ((xmax - xmin) * diff) / (2 * K)

greek_max_mins = {}
for k, (xmin, xmax) in param_bounds.items():
    greek_label = 'd' + str(k)

    if k in ["theta", "v0"] and greek_label in labels.columns: #En extra term i kedjeregeln krävs för att ta hänsyn till sqrt-transformationen
        labels[greek_label] = scale_greek_target(labels[greek_label], xmin, xmax, df_scaler['K']) * (2 * sqrt_copy[k])
    elif greek_label in labels.columns:
        labels[greek_label] = scale_greek_target(labels[greek_label], xmin, xmax, df_scaler['K'])
    
    if greek_label in labels.columns:
        greek_max_mins[greek_label] = [labels[greek_label].max()]
        greek_max_mins[greek_label].append(labels[greek_label].min())


#Split skalat data
X_train, X_val, y_train, y_val = train_test_split(inputs, labels, test_size=0.1, random_state=42)

train_dataset = TensorDataset(
    torch.tensor(X_train.values, dtype=torch.float32),
    torch.tensor(y_train.values, dtype=torch.float32)
    )
val_dataset = TensorDataset(
    torch.tensor(X_val.values, dtype=torch.float32),
    torch.tensor(y_val.values, dtype=torch.float32)
    )

train_loader = DataLoader(train_dataset, batch_size=256, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=2048)


**Weights for Greek loss**

In [17]:
greek_w = {}
raw_weights = []
keys_in_order = []

# 1. Iterate through both the key and the [max, min] list correctly
for k, (max_val, min_val) in greek_max_mins.items():
    
    # Calculate range
    val_range = max_val - min_val
    
    # Calculate raw weight
    raw_weight = 1.0 / (val_range**2 + 1e-8)
    
    raw_weights.append(raw_weight)
    keys_in_order.append(k)

# 2. Convert the Python list to a PyTorch tensor BEFORE doing math
raw_weights_tensor = torch.tensor(raw_weights, dtype=torch.float32)

# 3. Normalize the tensor
normalized_weights_tensor = raw_weights_tensor / torch.mean(raw_weights_tensor)

# 4. Map them back to the dictionary for easy lookup
for i, k in enumerate(keys_in_order):
    # .item() extracts the standard Python float from the 0D tensor
    greek_w[f'{k}'] = normalized_weights_tensor[i].item() 

print(greek_w)


{'dtheta': 0.38172540068626404, 'dsigma': 1.2735569477081299, 'drho': 2.828242778778076, 'dkappa': 0.004637331236153841, 'dv0': 0.5118373036384583}


In [18]:
import torch

greek_w = {}
raw_weights = []
keys_in_order = []

# Iterate over the greek names (keys)
for k in greek_max_mins.keys():
    greek_label = k
    
    # Only include targets that exist in your dataset
    if greek_label in y_train.columns:
        print(greek_label)
        
        # Variance from training data
        col_variance = y_train[greek_label].var()
        
        # Convert to float (safe for torch later)
        col_variance = float(col_variance)
        
        # Inverse variance weighting
        raw_weight = 1.0 / (col_variance + 1e-8)
        
        raw_weights.append(raw_weight)
        keys_in_order.append(greek_label)

# Convert to tensor
raw_weights_tensor = torch.tensor(raw_weights, dtype=torch.float32)

# Normalize so mean = 1
normalized_weights_tensor = raw_weights_tensor / torch.mean(raw_weights_tensor)

# Map back to dictionary
for i, key in enumerate(keys_in_order):
    greek_w[key] = normalized_weights_tensor[i].item()

print("Inverse Variance Weights (from y_train):\n", greek_w)

dtheta
dsigma
drho
dkappa
dv0
Inverse Variance Weights (from y_train):
 {'dtheta': 0.04477443918585777, 'dsigma': 1.7790203094482422, 'drho': 2.9474892616271973, 'dkappa': 0.05845973268151283, 'dv0': 0.1702565997838974}


**Training & Validation Steps**

In [19]:
# Bestämmer vilka greeks vi plockar ut för pred_grads och y_grads som ska användas i lossfunktionen.
relevant_greeks = ['kappa', 'v0', 'rho', 'sigma', 'theta']

# y och full_grad bygger på dataset med icke-matchande kolumnordning     
input_columns = list(inputs.columns)
output_columns = list(labels.columns)

index_in_input = torch.as_tensor([input_columns.index(k) for k in relevant_greeks], device=device)
index_in_output = torch.as_tensor([output_columns.index(f'd{k}') for k in relevant_greeks], device=device)

weights_list = [greek_w[f'd{k}'] for k in relevant_greeks]
greek_weights_tensor = torch.tensor(weights_list, dtype=torch.float32, device=device)

In [20]:
def train_step(model, criterion, batch: tuple, optimizer, weights):
        
        x, y = batch

        x.requires_grad_(True) # "håll koll på gradienter"

        optimizer.zero_grad()

        # Price loss
        pred_price = model(x)
        y_price = y[:, 0:1]
        l_p = criterion(pred_price, y_price)


        # Compute gradients (Greeks)
        full_grads = torch.autograd.grad(
            outputs=pred_price,
            inputs=x,
            grad_outputs=torch.ones_like(pred_price),
            create_graph=True, # OBS detta är inte för att få greeks utan för att beräkna gradienten av greeksen för optimeraren!!
            # retain_graph=True, är sant automatiskt
            only_inputs=True
        )[0]

        # Greek loss
        pred_grads = full_grads[:, index_in_input]
        y_grads = y[:, index_in_output]
        mse_per_greek = torch.mean((pred_grads - y_grads)**2, dim=0)
        l_g = torch.sum(mse_per_greek * greek_weights_tensor)
            
        
        # Get greek weights
        lambda_p = weights['lambda_p']
        lambda_g = weights['lambda_g']

        # Loss function
        loss = (lambda_p * l_p) + (lambda_g * l_g)

        # Compute loss gradients and update parameters
        loss.backward()
        optimizer.step()

        return {"loss": loss.item(), "l_p": l_p.item(), "l_g": l_g.item()}

In [21]:
def val_step(model, criterion, batch: Tuple, weights):

    x, y = batch

    x.requires_grad_(True)
  
    # Price loss
    pred_price = model(x)
    y_price = y[:, 0:1]
    l_p = criterion(pred_price, y_price)


    # Compute gradients (Greeks)
    full_grads = torch.autograd.grad(
            outputs=pred_price,
            inputs=x,
            grad_outputs=torch.ones_like(pred_price),
            create_graph=False, # Behövs ej vid test/validering (se komentar för training_step)
            retain_graph=False
        )[0]
        
    # Greek loss
    pred_grads = full_grads[:, index_in_input]
    y_grads = y[:, index_in_output]
    mse_per_greek = torch.mean((pred_grads - y_grads)**2, dim=0)
    l_g = torch.sum(mse_per_greek * greek_weights_tensor)
        

    # Get loss weights
    lambda_p = weights['lambda_p']
    lambda_g = weights['lambda_g']

    # Loss function
    loss = (lambda_p * l_p) + (lambda_g * l_g)

    return {"loss": loss.item(), "l_p": l_p.item(), "l_g": l_g.item()}

**HYPERPARAMETERS**

In [22]:
# Loss weights
lambda_p = 1
lambda_g = 1

# Model parameters 
nr_input_dim = 8
nr_hidden_layers = 7
nr_neurons = 250

# Optimizer settings
learning_rate = 3e-4
weight_decay = 1e-5
scheduler_patience = 7
scheduler_factor = 0.5

# Training loop
EPOCHS = 450
patience_early_stopping = 450

In [23]:
# initialisera modellen till device
model = DDN(input_dim=nr_input_dim, hidden_layers=nr_hidden_layers, neurons=nr_neurons).to(device)

# Loss + Optimizer
criterion = nn.MSELoss()
optimizer = optim.AdamW(model.parameters(), lr=learning_rate, weight_decay=weight_decay)
# Överväg att sänka lr och ska vi ha L-BFGS???
#scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=8, factor=0.5) # bestämer hur och när vi ska ändra lr 
scheduler = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(optimizer, T_0=150, T_mult=1, eta_min=1e-6)

**TRAINING LOOP**

In [24]:

# Path for saving best model state
import heston_project
PKG_DIR = Path(heston_project.__file__).resolve().parent
SAVE_DIR = PKG_DIR / "models" / "saved"
out_path = SAVE_DIR / "DDN_PdivK_new_weights_p1.pth"

# Tracker for early stopping
counter = 0

# Loss dicts
train_losses = {'total_train_loss': [], 'lp_train': [], 'lg_train': []}
val_losses = {'total_val_loss': [], 'lp_val': [], 'lg_val': []}

best_val_loss = 1000

# Loss weights
weights_dict = {'lambda_p': lambda_p, 'lambda_g': lambda_g}

print(f'Starting Training on {device}...')
start_time = time.time()        ##########


for epoch in range(EPOCHS):
    model.train()
    running_tot_loss, running_lp_loss, running_lg_loss = 0.0, 0.0, 0.0

    for batch_X, batch_y in train_loader:
        batch_X, batch_y = batch_X.to(device), batch_y.to(device)

        losses = train_step(model, criterion, batch = (batch_X, batch_y), optimizer = optimizer, weights = weights_dict)

        running_tot_loss += losses['loss']
        running_lp_loss += losses['l_p']
        running_lg_loss += losses['l_g']
    
    avg_train_total_loss = running_tot_loss/len(train_loader)
    avg_train_lp_loss = running_lp_loss/len(train_loader)
    avg_train_lg_loss = running_lg_loss/len(train_loader)

    train_losses['total_train_loss'].append(avg_train_total_loss)
    train_losses['lp_train'].append(avg_train_lp_loss)
    train_losses['lg_train'].append(avg_train_lg_loss)

    model.eval()
    running_tot_loss, running_lp_loss, running_lg_loss = 0.0, 0.0, 0.0

    # OBS ANVÄND ABSOLUT INTE TORCH NO GRAD
    for batch_X, batch_y in val_loader:
            batch_X, batch_y = batch_X.to(device), batch_y.to(device)

            losses = val_step(model, criterion = criterion, batch = (batch_X, batch_y), weights = weights_dict)

            running_tot_loss += losses['loss']
            running_lp_loss += losses['l_p']
            running_lg_loss += losses['l_g']

    
    avg_val_total_loss = running_tot_loss/len(val_loader)
    avg_val_lp_loss = running_lp_loss/len(val_loader)
    avg_val_lg_loss = running_lg_loss/len(val_loader)


    val_losses['total_val_loss'].append(avg_val_total_loss)
    val_losses['lp_val'].append(avg_val_lp_loss)
    val_losses['lg_val'].append(avg_val_lg_loss)

    # Uppdatera Learning Rate
    scheduler.step()

    if avg_val_total_loss < best_val_loss:
        best_val_loss = avg_val_total_loss
        best_loss_price = avg_val_lp_loss
        best_loss_greeks = avg_val_lg_loss
        torch.save(model.state_dict(), out_path) # Save the best version
        counter = 0 # Återställ!!!
    else:
        counter += 1
        if counter >= patience_early_stopping:
            print(f"Early stopping triggered at epoch {epoch}")
            break
    
    if (epoch + 1) % 5 == 0:
        current_lr = optimizer.param_groups[0]['lr']
        print(f"Epoch [{epoch+1:3d}/{EPOCHS}] | LR: {current_lr:.2e}")
        print(f"  Train -> Tot: {avg_train_total_loss:.4e} | Price: {avg_train_lp_loss:.4e} | Greeks: {avg_train_lg_loss:.4e}")
        print(f"  Val   -> Tot: {avg_val_total_loss:.4e} | Price: {avg_val_lp_loss:.4e} | Greeks: {avg_val_lg_loss:.4e}")
        print("-" * 70)

print("*" * 70)
print(f"Best Val Loss -> Tot: {best_val_loss:.4e} | Price: {best_loss_price:.4e} | Greeks: {best_loss_greeks:.4e}")
print("*" * 70)

end_time = time.time()
print(f"Training Finished. Total Time: {end_time - start_time:.2f} seconds")

loss_df = pd.DataFrame({
    "epoch": range(1, len(train_losses["total_train_loss"]) + 1),
    "train_total_loss": train_losses["total_train_loss"],
    "train_lp_loss": train_losses["lp_train"],
    "train_lg_loss": train_losses["lg_train"],
    "val_total_loss": val_losses["total_val_loss"],
    "val_lp_loss": val_losses["lp_val"],
    "val_lg_loss": val_losses["lg_val"],
})

loss_df.to_csv("losses/loss_data/DDN_PdivK_loss_history_new_weights_p1.csv", index=False)

# Best Val Loss -> Tot: 1.6824e-07 | Price: 5.1196e-09 | Greeks: 1.1705e-07



Starting Training on cpu...
Epoch [  5/450] | LR: 2.99e-04
  Train -> Tot: 3.5419e-05 | Price: 1.2540e-05 | Greeks: 2.2878e-05
  Val   -> Tot: 2.7571e-05 | Price: 6.8796e-06 | Greeks: 2.0691e-05
----------------------------------------------------------------------
Epoch [ 10/450] | LR: 2.97e-04
  Train -> Tot: 1.5483e-05 | Price: 5.7552e-06 | Greeks: 9.7277e-06
  Val   -> Tot: 9.9666e-06 | Price: 2.6128e-06 | Greeks: 7.3538e-06
----------------------------------------------------------------------
Epoch [ 15/450] | LR: 2.93e-04
  Train -> Tot: 9.6099e-06 | Price: 3.8819e-06 | Greeks: 5.7280e-06
  Val   -> Tot: 5.6698e-05 | Price: 3.1264e-05 | Greeks: 2.5433e-05
----------------------------------------------------------------------
Epoch [ 20/450] | LR: 2.87e-04
  Train -> Tot: 1.0286e-05 | Price: 3.9193e-06 | Greeks: 6.3671e-06
  Val   -> Tot: 6.9756e-06 | Price: 1.6632e-06 | Greeks: 5.3124e-06
----------------------------------------------------------------------
Epoch [ 25/450] | LR